In [ ]:
!pip install -q kagglehub

import kagglehub
kagglehub.login()

In [ ]:
images_path = kagglehub.dataset_download("khanfashee/nih-chest-x-ray-14-224x224-resized")
print("Images downloaded to:", images_path)

In [ ]:
import os

csv_candidates = [os.path.join(root, f) for root, _, files in os.walk(images_path) for f in files if f.endswith(".csv")]

if csv_candidates:
    print("Found labels bundled with the image dataset:", csv_candidates)
    labels_csv_path = csv_candidates[0]
else:
    print("No labels CSV bundled with the resized images — fetching Data_Entry_2017.csv only "
          "(not the full 42GB original dataset).")
    labels_dir = kagglehub.dataset_download("nih-chest-xrays/data", path="Data_Entry_2017.csv")
    labels_csv_path = labels_dir if labels_dir.endswith(".csv") else os.path.join(labels_dir, "Data_Entry_2017.csv")
    print("Labels downloaded to:", labels_csv_path)

img_dir_candidates = [root for root, dirs, files in os.walk(images_path) if any(f.lower().endswith((".png", ".jpg")) for f in files)]
IMAGES_DIR = img_dir_candidates[0]
print("Images folder:", IMAGES_DIR)

In [ ]:
import pandas as pd
import numpy as np

full_labels_df = pd.read_csv(labels_csv_path)
print("Full dataset:", full_labels_df.shape)

N_SUBSET = 25000  # keeps a Colab session comfortably within free-tier limits
labels_df = full_labels_df.sample(n=N_SUBSET, random_state=42).reset_index(drop=True)
print("Subset:", labels_df.shape)

all_labels = labels_df["Finding Labels"].str.split("|").explode()
label_counts = all_labels.value_counts()
print(label_counts)

In [ ]:
CONDITIONS = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass",
    "Nodule", "Pneumonia", "Pneumothorax", "Consolidation", "Edema",
    "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia"
]

missing = [c for c in CONDITIONS if c not in label_counts.index]
if missing:
    print("WARNING: these condition names don't match the dataset's labels:", missing)
    print("Available labels:", sorted(label_counts.index.tolist()))
else:
    print("All 14 condition names match.")

def labels_to_vector(finding_labels_str):
    labels = finding_labels_str.split("|")
    return np.array([1.0 if c in labels else 0.0 for c in CONDITIONS], dtype=np.float32)

label_matrix = np.stack(labels_df["Finding Labels"].apply(labels_to_vector).to_numpy())

print()
for i, c in enumerate(CONDITIONS):
    print(f"{c}: {int(label_matrix[:, i].sum())} positive examples in subset")

In [ ]:
from sklearn.model_selection import train_test_split

image_indices = labels_df["Image Index"].to_numpy()
train_idx, val_idx = train_test_split(np.arange(len(image_indices)), test_size=0.2, random_state=42)

train_files = image_indices[train_idx]
val_files = image_indices[val_idx]
train_labels = label_matrix[train_idx]
val_labels = label_matrix[val_idx]

print(f"Train: {len(train_files)}, Val: {len(val_files)}")

In [ ]:
import tensorflow as tf

IMG_SIZE = 224

def load_image(filename, label):
    path = tf.strings.join([IMAGES_DIR, filename], separator="/")
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])  # safety net if any file isn't exactly 224x224
    image = image / 255.0
    return image, label

def make_dataset(files, labels, shuffle=False, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices((files, labels))
    if shuffle:
        ds = ds.shuffle(len(files), seed=42)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_files, train_labels, shuffle=True)
val_ds = make_dataset(val_files, val_labels)

for imgs, labs in train_ds.take(1):
    print("batch image shape:", imgs.shape, "batch label shape:", labs.shape)

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121

base_model = DenseNet121(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False  # phase 1: frozen base

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(CONDITIONS), activation="sigmoid")(x)
model = models.Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(multi_label=True, name="auc")],
)
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Checkpointing every epoch is cheap insurance against a Colab disconnect —
# if the session drops mid-run, you don't lose the whole training run.
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint("checkpoint_phase1.keras", save_best_only=True, monitor="val_loss"),
]

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    shuffle=False,
    callbacks=callbacks,
)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:  # keep most of DenseNet frozen; only fine-tune the top blocks
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # much lower — avoid wrecking pretrained weights
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(multi_label=True, name="auc")],
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint("checkpoint_phase2.keras", save_best_only=True, monitor="val_loss"),
]

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    shuffle=False,
    callbacks=callbacks,
)

In [ ]:
from sklearn.metrics import roc_auc_score

y_true = val_labels
y_pred = model.predict(val_ds, verbose=0)

print(f"{'Condition':<20} {'AUC':>6} {'Positives in val':>18}")
aucs = []
for i, c in enumerate(CONDITIONS):
    n_pos = int(y_true[:, i].sum())
    if n_pos == 0 or n_pos == len(y_true):
        print(f"{c:<20} {'N/A':>6} {n_pos:>18}")
        continue
    auc = roc_auc_score(y_true[:, i], y_pred[:, i])
    aucs.append(auc)
    print(f"{c:<20} {auc:>6.3f} {n_pos:>18}")

print(f"\nMean AUC across conditions with enough data: {np.mean(aucs):.3f}")

In [ ]:
import json

os.makedirs("model_artifacts", exist_ok=True)
model.save("model_artifacts/v2_densenet_transfer.keras")

with open("model_artifacts/condition_names.json", "w") as f:
    json.dump(CONDITIONS, f, indent=2)

print("Artifacts written:")
!ls -la model_artifacts

In [ ]:
import shutil

shutil.make_archive("v2_imaging_artifacts", "zip", "model_artifacts")

from google.colab import files
files.download("v2_imaging_artifacts.zip")